In [1]:
import math

t_new = 131_072
t_pretrain = 8192
theta_p = 500_000

theta_n = theta_p ** (math.log(t_new / (2*math.pi), t_pretrain / (2*math.pi)))
theta_n

79760692.221776

In [ ]:
from recipes_2.efficient_fce import FusedProjectionPlusCrossEntropyLoss



In [108]:
import numpy as np
from typing import List
import torch

def slice_seq_lengths(
                      seq_lengths: List[List[int]],
                      slice_: slice) -> List[List[int]]:
  cpu = torch.device("cpu")
  seq_lengths     = torch.tensor(sum(seq_lengths, []), device=cpu)
  seq_len_cumsum  = torch.cumsum(seq_lengths, dim=0).to(cpu)

  end_mask = (seq_len_cumsum >= slice_.stop).nonzero()

  if len(end_mask) > 0:
    end_idx = end_mask[0][0]
    end_seq_len =  seq_lengths[end_idx] - (seq_len_cumsum[end_idx] - slice_.stop)
    seq_lengths[end_idx] = end_seq_len
  else:
    end_idx = len(seq_len_cumsum) - 1  


  start_idx = (seq_len_cumsum >= slice_.start).nonzero()[0][0]
  start_seq_len = seq_len_cumsum[start_idx] - slice_.start
  if start_seq_len == 0:
    start_idx += 1
  else:
    seq_lengths[start_idx] = start_seq_len
  

  return [seq_lengths[start_idx:end_idx + 1].tolist()]

In [109]:
og_lens = [[3, 4, 5, 6, 7]]

slice_ = slice(3, 1000)

sliced_seq_lens = slice_seq_lengths(og_lens, slice_)

print(sliced_seq_lens)
print("SUM :", sum(sliced_seq_lens[0]))


[[4, 5, 6, 7]]
SUM : 22


In [94]:
slice_ = slice(1)

In [95]:
print(slice_.start)

None


In [96]:
print(slice_.stop)

1
